# Comprehensive Exploratory Data Analysis (EDA)

This notebook conducts a deep, granular analysis of every dataset provided in the CreditNirvana problem statements. We will analyze missing values, correlations, feature distributions, temporal trends, and target relationships across all three problem domains.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
pd.set_option('display.max_columns', None)

base_path = r"../Dataset/data"

## Part 1: Shared Data Analysis
### 1.1 Accounts Data
Exploring the distribution of lenders, loan products, delinquency buckets, and balances.

In [ ]:
accounts = pd.read_csv(os.path.join(base_path, "shared", "accounts.csv"))
print(f"Total Accounts: {len(accounts)}")
display(accounts.head(3))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.countplot(data=accounts, x='portfolio', ax=axes[0,0])
axes[0,0].set_title('Portfolio Distribution')

sns.countplot(data=accounts, x='bucket_start', ax=axes[0,1], order=['X', '1-30', '31-60', '61-90', '90+'])
axes[0,1].set_title('Delinquency Buckets')

sns.histplot(data=accounts, x='outstanding', bins=50, ax=axes[1,0], log_scale=True)
axes[1,0].set_title('Log-Scaled Outstanding Balance')

sns.countplot(data=accounts, x='income_type', ax=axes[1,1])
axes[1,1].set_title('Income Type')
plt.tight_layout()
plt.show()

### 1.2 Dial Attempts & Network Responses
Analyzing calling patterns, network statuses, and durations.

In [ ]:
dial_attempts = pd.read_csv(os.path.join(base_path, "shared", "dial_attempts.csv"))
dial_attempts['attempt_ts'] = pd.to_datetime(dial_attempts['attempt_ts'])
print(f"Total Dial Attempts: {len(dial_attempts)}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.countplot(data=dial_attempts, y='network_response', ax=axes[0])
axes[0].set_title('Network Responses')

# Time of day analysis
dial_attempts['hour'] = dial_attempts['attempt_ts'].dt.hour
sns.histplot(data=dial_attempts, x='hour', bins=24, ax=axes[1], discrete=True)
axes[1].set_title('Dial Attempts by Hour of Day')
plt.show()

print("Talk Duration Statistics (for answered calls):")
display(dial_attempts[dial_attempts['talk_duration_s'] > 0]['talk_duration_s'].describe())

### 1.3 Field Visits
Exploring the outcomes and dwell times of field visits.

In [ ]:
field_visits = pd.read_csv(os.path.join(base_path, "shared", "field_visits.csv"))
print(f"Total Field Visits: {len(field_visits)}")

plt.figure(figsize=(10, 6))
sns.countplot(data=field_visits, y='outcome', order=field_visits['outcome'].value_counts().index)
plt.title('Field Visit Outcomes')
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=field_visits, x='dwell_s', y='outcome')
plt.xlim(0, 1500)
plt.title('Dwell Time (seconds) by Visit Outcome')
plt.show()

## Part 2: PS1 - Fake PTP Detection
Deep dive into Promises to Pay, transcripts, and payment outcomes.

In [ ]:
ptps = pd.read_csv(os.path.join(base_path, "ps1_fake_ptp", "ptps.csv"))
payments = pd.read_csv(os.path.join(base_path, "shared", "payments.csv"))
annotations = pd.read_csv(os.path.join(base_path, "ps1_fake_ptp", "annotated_ptp_sample.csv"))

ptps['captured_ts'] = pd.to_datetime(ptps['captured_ts'])
ptps['promised_date'] = pd.to_datetime(ptps['promised_date'])
payments['payment_ts'] = pd.to_datetime(payments['payment_ts'])

# Determine True Kept Rate
ptp_payments = pd.merge(ptps, payments, on='account_id', how='left')
valid_payments = ptp_payments[
    (ptp_payments['payment_ts'] >= ptp_payments['captured_ts']) &
    (ptp_payments['payment_ts'] <= ptp_payments['promised_date'] + pd.Timedelta(days=3))
]
payment_sums = valid_payments.groupby('ptp_id')['amount'].sum().reset_index().rename(columns={'amount': 'total_paid'})
ptps = pd.merge(ptps, payment_sums, on='ptp_id', how='left').fillna({'total_paid': 0})
ptps['kept'] = (ptps['total_paid'] >= (ptps['promised_amount'] * 0.9)).astype(int)

# Analysis on Annotated set
ptp_annotated = pd.merge(ptps, annotations, on='ptp_id', how='inner')
plt.figure(figsize=(12, 6))
sns.barplot(data=ptp_annotated, y='annotated_type', x='kept', ci=None)
plt.title('Actual PTP Kept Rate by Annotated Category')
plt.xlabel('Keep Rate (Probability)')
plt.show()

# Amount comparison
plt.figure(figsize=(8, 6))
sns.scatterplot(data=ptps, x='promised_amount', y='total_paid', hue='kept', alpha=0.5)
plt.title('Promised Amount vs Actual Paid Amount')
plt.plot([0, ptps['promised_amount'].max()], [0, ptps['promised_amount'].max()], 'r--')
plt.show()

## Part 3: PS2 - Right-Party Contact
Exploring phone network health and verified data.

In [ ]:
phones = pd.read_csv(os.path.join(base_path, "ps2_right_party_contact", "phones.csv"))
verified = pd.read_csv(os.path.join(base_path, "ps2_right_party_contact", "verified_contact_points.csv"))

phone_attempts = dial_attempts.groupby('phone_id').agg(
    total_attempts=('attempt_id', 'count'),
    switched_off_ratio=('network_response', lambda x: (x == 'switched_off').mean()),
    answered_ratio=('talk_duration_s', lambda x: (x > 0).mean())
).reset_index()

ps2_data = pd.merge(verified, phone_attempts, on='phone_id', how='inner')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.boxplot(data=ps2_data, y='verified_status', x='switched_off_ratio', ax=axes[0])
axes[0].set_title('Switched Off Ratio by Verified Status')

sns.boxplot(data=ps2_data, y='verified_status', x='answered_ratio', ax=axes[1])
axes[1].set_title('Answered Ratio by Verified Status')
plt.tight_layout()
plt.show()

## Part 4: PS3 - Address Geocoder
Detailed view of GPS trails, precision, and accuracy.

In [ ]:
gps = pd.read_csv(os.path.join(base_path, "ps3_geocoder", "visit_gps_points.csv"))
baseline = pd.read_csv(os.path.join(base_path, "ps3_geocoder", "baseline_geocodes.csv"))
surveyed = pd.read_csv(os.path.join(base_path, "ps3_geocoder", "surveyed_addresses.csv"))

# Merge surveyed with baseline
ps3_eval = pd.merge(surveyed, baseline, on='address_id', how='inner')
ps3_eval['baseline_error'] = np.sqrt((ps3_eval['geocoder_x'] - ps3_eval['surveyed_x'])**2 + (ps3_eval['geocoder_y'] - ps3_eval['surveyed_y'])**2)

plt.figure(figsize=(10, 5))
sns.histplot(data=ps3_eval, x='baseline_error', bins=30, kde=True)
plt.title('Baseline Geocoder Error Distribution (m)')
plt.xlabel('Distance Error (meters)')
plt.show()

# Examine GPS accuracy reported by devices
plt.figure(figsize=(10, 5))
sns.histplot(data=gps, x='accuracy_m', bins=50, log_scale=(False, True))
plt.title('Device Reported GPS Accuracy Distribution (Log Scale)')
plt.xlabel('Accuracy (meters)')
plt.show()

## Conclusion
This detailed EDA reveals that:
1. PTP kept rates vary wildly depending on the annotated psychological context (escape vs genuine). NLP is mandatory.
2. Right-party contacts can be strongly inferred by historical network responses (e.g., switched-off ratios).
3. The baseline geocoder has high variance, while the device GPS has heavy tails of inaccuracy which must be filtered (accuracy < 30m).